# B00 — Inventaris dataset untuk benchmark empat dataset

Jalankan dari atas ke bawah di **Colab dengan CPU**. Notebook ini memeriksa arsip dan CSV sebelum pelatihan. Tidak melakukan training, menebak label, atau mengganti CSV/checkpoint eksperimen sebelumnya.

**Rencana:** VGG16, ResNet50, InceptionV3, RegNetY-8GF; DenseNet121, MobileNetV2, MobileNetV3-Small, NASNetMobile. Varian ini adalah usulan awal untuk memperjelas nama keluarga model dari arahan pembimbing. Status Teacher/Student adalah peran eksperimen, bukan jaminan ukuran model. Semua kandidat dilatih secara supervised terlebih dahulu; MoE/KD menyusul setelah seleksi melalui validation.

**Per dataset:** split pasien terpisah, seed split 2026; seed training awal 42; Precision/Recall/F1 kelas positif dan macro-F1; Accuracy/ROC-AUC/AP tambahan. Threshold utama 0,5. Pilih checkpoint dengan validation macro-F1; tie-break validation recall kelas positif. Test hanya untuk evaluasi akhir setelah pilihan dibekukan. Hasil fase satu seed bersifat pendahuluan. Rasio awal 70/15/15 berdasarkan pasien, harus diperiksa kecukupan kelasnya sebelum dibekukan. CSV split dibuat setelah audit cohort, bukan oleh notebook inventaris ini.

**Batas interpretasi:** training pada CMMD lalu test CMMD adalah evaluasi dalam dataset, bukan validasi eksternal. Label berbasis BI-RADS pada INbreast tidak identik dengan diagnosis patologi. MIAS normal tidak otomatis disamakan dengan benign. CMMD baru harus dipetakan menurut patient dan sisi; cohort lama tidak diwariskan otomatis.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Lokasi data
Jika folder `datasets` ada langsung di MyDrive, pengaturan awal sudah sesuai. Jika berada di `Research/datasets`, pencarian kedua juga mencakupnya. Isi path override jika ada beberapa salinan arsip dengan nama sama. Pencarian dibatasi pada folder di bawah ini; notebook tidak menelusuri seluruh Drive.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json, io, zipfile, hashlib, platform, importlib.util
import pandas as pd
from IPython.display import display

DATASET_ROOTS = [Path('/content/drive/MyDrive/datasets'),
                 Path('/content/drive/MyDrive/Research/datasets')]
ARCHIVE_OVERRIDES = {'cmmd': '', 'mias': '', 'inbreast': ''}
ARCHIVE_NAMES = {'cmmd': 'TheChineseMammographyDatabase.zip',
                 'mias': 'miasdbv1.21.zip', 'inbreast': None}
# Set the exact path if the automatic candidates do not locate the CSV.
CBIS_ALL_CSV = ''
CBIS_CANDIDATES = [
    Path('/content/drive/MyDrive/Research/processed/cbis_all.csv'),
    Path('/content/drive/MyDrive/processed/cbis_all.csv'),
    Path('/content/drive/MyDrive/Research/datasets/processed/cbis_all.csv'),
    Path('/content/drive/MyDrive/datasets/processed/cbis_all.csv'),
]
OUTPUT_BASE = Path('/content/drive/MyDrive/Research/outputs/benchmark_4datasets')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
MAX_METADATA_BYTES = 20 * 1024 * 1024
MAX_METADATA_FILES = 15
MAX_HEADER_SAMPLES = 6

report = {'run_id': RUN_ID, 'python': platform.python_version(),
          'archives': {}, 'cbis': {}, 'notes': [
              'Archive inventory is not image/label validation.',
              'No train/validation/test manifests are generated in this notebook.',
              'Archive index hash covers member names/size/CRC, not a full archive SHA256.'
          ]}
print('CPU is sufficient. No GPU is required for this notebook.')
print('Dataset folders:', {str(p): p.exists() for p in DATASET_ROOTS})

## 2. Cari arsip
Nama INbreast dicari dari berkas ZIP dengan kata `inbreast`. Jika data sudah diekstrak atau nama ZIP berbeda, isi override. Status MISSING/AMBIGUOUS tetap dilaporkan; notebook tidak memilih salinan secara sembarang.

In [ ]:
def bounded_zip_search(root):
    if not root.is_dir():
        return []
    # Root plus two folder levels only; no recursive image-directory traversal.
    found = []
    for pattern in ('*.zip', '*/*.zip', '*/*/*.zip', '*.ZIP', '*/*.ZIP', '*/*/*.ZIP'):
        found.extend(root.glob(pattern))
    return sorted(set(p.resolve() for p in found))

available_zips = sorted(set(p for root in DATASET_ROOTS for p in bounded_zip_search(root)))
print('ZIP candidates:')
for p in available_zips:
    print(p)

archive_paths = {}
for dataset, expected in ARCHIVE_NAMES.items():
    override = ARCHIVE_OVERRIDES[dataset].strip()
    if override:
        candidates = [Path(override)] if Path(override).is_file() else []
    else:
        candidates = [p for p in available_zips if
                      (p.name.casefold() == expected.casefold() if expected
                       else 'inbreast' in p.name.casefold())]
    if len(candidates) == 1:
        archive_paths[dataset] = candidates[0]
        print(dataset, 'FOUND', candidates[0])
    else:
        status = 'MISSING' if not candidates else 'AMBIGUOUS'
        report['archives'][dataset] = {'status': status, 'candidates': [str(p) for p in candidates]}
        print(dataset, status, 'Set ARCHIVE_OVERRIDES if needed.')

## 3. Struktur, metadata, dan contoh header citra
Notebook membaca daftar isi ZIP serta berkas metadata kecil, tanpa mengekstrak semua citra. Pratinjau label dibatasi 20 baris dan tidak menetapkan mapping. Ringkasan semua anggota ZIP disimpan pada tahap akhir.

Jika MIAS ternyata berisi arsip bersarang (`.gz`, `.tar`, atau ZIP lain), isinya dilaporkan untuk dibaca pada tahap berikutnya. Tidak menganggap versi MIAS sama dengan mini-MIAS. Pembacaan DICOM header bersifat opsional bila `pydicom` tersedia di runtime.

In [ ]:
member_tables = {}
def inspect_archive(path):
    result = {'path': str(path), 'status': 'INVENTORIED',
              'archive_bytes': path.stat().st_size, 'metadata': [], 'image_headers': []}
    with zipfile.ZipFile(path) as z:
        members = [m for m in z.infolist() if not m.is_dir()]
        if not members:
            raise ValueError('Empty ZIP archive')
        names = [m.filename for m in members]
        result['duplicate_member_names'] = len(names) - len(set(names))
        result['member_count'] = len(members)
        result['uncompressed_bytes'] = sum(m.file_size for m in members)
        result['extensions'] = dict(Counter(Path(m.filename).suffix.lower() or '(none)' for m in members))
        index = [{'member': m.filename, 'bytes': m.file_size, 'compressed_bytes': m.compress_size,
                  'crc32': f'{m.CRC:08x}'} for m in members]
        result['index_sha256'] = hashlib.sha256(json.dumps(index, sort_keys=True).encode()).hexdigest()
        result['member_preview'] = names[:30]
        # Prioritize metadata by extension and meaningful names, but never execute it.
        meta = [m for m in members if Path(m.filename).suffix.lower() in
                {'.csv', '.xlsx', '.xls', '.txt', '.md', '.json'} or
                any(word in Path(m.filename).name.lower() for word in ['readme', 'license', 'info.txt'])]
        meta.sort(key=lambda m: (Path(m.filename).suffix.lower() not in {'.csv','.xlsx','.xls'}, m.filename))
        result['metadata_candidates'] = [m.filename for m in meta]
        for m in meta[:MAX_METADATA_FILES]:
            item = {'member': m.filename, 'bytes': m.file_size}
            if m.file_size > MAX_METADATA_BYTES:
                item['status'] = 'SKIPPED_SIZE_LIMIT'
            else:
                try:
                    data = z.read(m)
                    suffix = Path(m.filename).suffix.lower()
                    if suffix == '.csv':
                        table = pd.read_csv(io.BytesIO(data), dtype=str, keep_default_na=False)
                        item.update(rows=len(table), columns=list(table.columns),
                                    preview=table.head(20).to_dict('records'))
                    elif suffix in {'.xls', '.xlsx'}:
                        sheets = pd.read_excel(io.BytesIO(data), sheet_name=None, dtype=str)
                        item['sheets'] = {name: {'rows': len(table), 'columns': list(table.columns),
                            'preview': table.fillna('').head(20).to_dict('records')}
                            for name, table in sheets.items()}
                    else:
                        item['text_preview'] = data[:16000].decode('utf-8', errors='replace')
                    item['status'] = 'READ'
                except Exception as exc:
                    item.update(status='READ_ERROR', error=f'{type(exc).__name__}: {exc}')
            result['metadata'].append(item)
        image_candidates = [m for m in members if Path(m.filename).suffix.lower() in
                            {'.jpg','.jpeg','.png','.pgm','.tif','.tiff','.dcm'}]
        for m in image_candidates[:MAX_HEADER_SAMPLES]:
            item = {'member': m.filename}
            try:
                with z.open(m) as stream:
                    if Path(m.filename).suffix.lower() == '.dcm':
                        if importlib.util.find_spec('pydicom') is None:
                            item['status'] = 'DICOM_HEADER_PENDING_PYDICOM'
                        else:
                            import pydicom
                            ds = pydicom.dcmread(stream, stop_before_pixels=True)
                            item['header'] = {key: str(getattr(ds, key, '')) for key in
                                ['PatientID','ImageLaterality','Laterality','ViewPosition',
                                 'Rows','Columns','BitsStored','PhotometricInterpretation','SeriesInstanceUID']}
                            item['status'] = 'HEADER_READ'
                    else:
                        from PIL import Image
                        image = Image.open(stream)
                        item.update(status='HEADER_READ', width=image.width, height=image.height,
                                    mode=image.mode, format=image.format)
            except Exception as exc:
                item.update(status='HEADER_ERROR', error=f'{type(exc).__name__}: {exc}')
            result['image_headers'].append(item)
    return result, pd.DataFrame(index)

for dataset, path in archive_paths.items():
    print('Inspecting', dataset, path.name)
    try:
        result, table = inspect_archive(path)
        report['archives'][dataset] = result
        member_tables[dataset] = table
        print('Members:', result['member_count'], 'Extensions:', result['extensions'])
        display(table.head(12))
        for item in result['metadata']:
            print('METADATA:', item['member'], item['status'])
            if 'columns' in item:
                print('Rows:', item['rows'], 'Columns:', item['columns'])
                display(pd.DataFrame(item['preview']).head(5))
            if 'sheets' in item:
                for name, sheet in item['sheets'].items():
                    print('Sheet:', name, 'Rows:', sheet['rows'], 'Columns:', sheet['columns'])
                    display(pd.DataFrame(sheet['preview']).head(5))
            if 'text_preview' in item:
                print(item['text_preview'][:3000])
        display(pd.DataFrame(result['image_headers']))
    except Exception as exc:
        report['archives'][dataset] = {'path': str(path), 'status': 'INVENTORY_ERROR',
                                        'error': f'{type(exc).__name__}: {exc}'}
        print(report['archives'][dataset])

## 4. Audit `cbis_all.csv` dan split lama bila tersedia
`cbis_all.csv` dipertahankan sebagai daftar sampel. Tidak otomatis mengambil satu dari beberapa file bernama sama. Audit ini memeriksa isi CSV, bukan keberadaan atau piksel semua JPEG di Drive.

In [ ]:
def audit_cbis(path):
    table = pd.read_csv(path, dtype={'patient_id': str})
    required = {'image_path', 'patient_id', 'cancer'}
    missing = required - set(table.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    if table[list(required)].isna().any().any():
        raise ValueError('Null path, patient ID, or label')
    if table.image_path.astype(str).str.strip().eq('').any() or table.patient_id.str.strip().eq('').any():
        raise ValueError('Empty path or patient ID')
    labels = pd.to_numeric(table.cancer, errors='raise')
    if not labels.isin([0, 1]).all():
        raise ValueError('Labels must be exactly 0 or 1; no rounding is allowed')
    if table.image_path.duplicated().any():
        raise ValueError('Duplicate image paths; investigate before making new splits')
    table['cancer'] = labels.astype(int)
    result = {'status': 'CSV_AUDITED', 'path': str(path),
              'sha256': hashlib.sha256(path.read_bytes()).hexdigest(),
              'images': len(table), 'patients': table.patient_id.nunique(),
              'positive_images': int(table.cancer.sum()),
              'negative_images': int((table.cancer == 0).sum()),
              'columns': list(table.columns), 'image_path_preview': table.image_path.head(3).tolist(),
              'image_files_checked': False}
    patient_labels = table.groupby('patient_id').cancer.max()
    result['patients_with_any_positive_image'] = int(patient_labels.sum())
    old_tr, old_va = path.parent/'cbis_train.csv', path.parent/'cbis_val.csv'
    if old_tr.is_file() and old_va.is_file():
        tr, va = pd.read_csv(old_tr, dtype={'patient_id':str}), pd.read_csv(old_va, dtype={'patient_id':str})
        result['old_split'] = {
            'train_images':len(tr), 'validation_images':len(va),
            'patient_overlap':len(set(tr.patient_id) & set(va.patient_id)),
            'image_path_overlap':len(set(tr.image_path) & set(va.image_path)),
            'union_paths_equal_all':set(tr.image_path) | set(va.image_path) == set(table.image_path),
            'note':'These are the historical train/validation CSVs, not a new held-out test.'}
    return result

if CBIS_ALL_CSV.strip():
    candidates = [Path(CBIS_ALL_CSV)] if Path(CBIS_ALL_CSV).is_file() else []
else:
    candidates = sorted(set(p.resolve() for p in CBIS_CANDIDATES if p.is_file()))
if len(candidates) == 1:
    try:
        report['cbis'] = audit_cbis(candidates[0])
    except Exception as exc:
        report['cbis'] = {'status':'AUDIT_ERROR', 'error':f'{type(exc).__name__}: {exc}'}
else:
    report['cbis'] = {'status':'MISSING' if not candidates else 'AMBIGUOUS',
                      'candidates':[str(p) for p in candidates],
                      'action':'Set CBIS_ALL_CSV to the exact Drive path and rerun this cell.'}
print(json.dumps(report['cbis'], indent=2))

## 5. Simpan laporan untuk ditinjau
Folder output memakai waktu UTC unik. File lama tidak ditimpa. Arsip laporan hanya berisi inventaris, contoh metadata, dan audit CSV; tidak berisi citra dataset.

Setelah selesai, unggah **`benchmark_dataset_inventory.zip`** yang dihasilkan ke percakapan. Dari laporan ini kita menetapkan label, unit pasien/payudara, dan pemilihan citra sebelum membuat split dan runner training. Status berhasil diinventaris tidak berarti dataset sudah lolos validasi ilmiah.

In [ ]:
output_dir = OUTPUT_BASE / 'inventory' / RUN_ID
output_dir.mkdir(parents=True, exist_ok=False)
report_path = output_dir / 'inventory_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
for dataset, table in member_tables.items():
    table.to_csv(output_dir/f'{dataset}_archive_members.csv', index=False)
summary = pd.DataFrame([{'dataset': key, 'status': value.get('status'),
                         'members':value.get('member_count'), 'archive_path':value.get('path')}
                        for key,value in report['archives'].items()])
summary.to_csv(output_dir/'archive_summary.csv', index=False)
bundle = output_dir/'benchmark_dataset_inventory.zip'
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(output_dir.iterdir()):
        if path.is_file() and path != bundle:
            z.write(path, arcname=path.name)
display(summary)
print('CBIS:', report['cbis'].get('status'))
print('Saved:', bundle)
print('Upload this report ZIP to continue preparing label mappings and patient splits.')

Unduh laporan (opsional; salinannya sudah tersimpan di Drive):

In [ ]:
from google.colab import files
files.download(str(bundle))